# Episode 13: Flash attention from scratch: PyTorch, Triton

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep013-flash-attention.ipynb)

We write attention without its table of scores, first in PyTorch on the Mac, then as a Triton kernel on a rented NVIDIA GPU, forwards and backwards, and measure it against the FlashAttention-2 inside PyTorch.

[The video](https://youtu.be/HEYzYWquKos) · [The episode's page](https://mvgpt.nenuadrian.com/ep013-flash-attention/) · [The docs](https://nenuadrian.github.io/mvGPT/flash-attention/) · [The code at `ep013/end`](https://github.com/nenuadrian/mvGPT/tree/ep013/end)

This notebook runs the episode's commands on its code, as the episode left it. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep013/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

The episode ran its GPU half on an RTX 4090, in bfloat16. Colab's free T4 has no bfloat16 in
hardware, so on it the benchmarks run in float16 and the Triton training run is left out;
an L4 or A100 runtime runs all of it.

## Attention by hand, and FlashAttention's algorithm in PyTorch

In [ ]:
!python -m pytest tests/test_attention.py -v

Memory on the CPU, and time on the GPU (the episode's was a laptop's, `--device mps`):

In [ ]:
!python attention_bench.py --device cpu 2>/dev/null
!python attention_bench.py --device cuda 2>/dev/null

## On the GPU: by hand against PyTorch's own

In [ ]:
!nvidia-smi

In [ ]:
import torch

# bfloat16 in hardware: an L4 or A100 runtime has it, the free T4 (compute capability 7.5) doesn't.
AMPERE = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
print("bfloat16 in hardware" if AMPERE else "no bfloat16 in hardware: these runs will be slow")

In [ ]:
DTYPE = "bfloat16" if AMPERE else "float16"
!python attention_bench.py --dtype {DTYPE} --batch 4 --lengths 1024 2048 4096 8192 16384 --methods by-hand pytorch --runs 10 2>/dev/null

## Our Triton kernel
The tests, then the forward pass, then forward and backward, against PyTorch's.

In [ ]:
!python -m pytest tests/test_flash.py -v

In [ ]:
!python attention_bench.py --dtype {DTYPE} --batch 4 --lengths 1024 2048 4096 8192 16384 --methods by-hand pytorch triton --runs 10 2>/dev/null
!python attention_bench.py --dtype {DTYPE} --batch 4 --backward --lengths 1024 2048 4096 8192 16384 --methods by-hand pytorch triton --runs 10 2>/dev/null

## Training with it
Five minutes each at 2,048 tokens, with PyTorch's attention and with ours. The episode's
batch was 12 sequences, on 24 GB; a 16 GB T4 takes 4.

In [ ]:
!python data.py --bpe  # TinyStories, the BPE merges, and the stories as BPE token ids

In [ ]:
BATCH = 12 if torch.cuda.get_device_properties(0).total_memory > 20e9 else 4
!torchrun --standalone --nproc_per_node 1 pretrain_ddp.py 5 --block 2048 --batch {BATCH} --attention pytorch --name flash-pytorch 2>/dev/null
if AMPERE:
    !torchrun --standalone --nproc_per_node 1 pretrain_ddp.py 5 --block 2048 --batch {BATCH} --attention triton --name flash-triton 2>/dev/null

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

Next: [episode 14, Batching: serving a GPT to many users at once](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep014-serving.ipynb)